In [14]:
import pandas as pd
import os
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from PIL import Image

dataset : https://www.kaggle.com/datasets/puneet6060/intel-image-classification?select=seg_train

Create the label files

In [10]:
dataset_train_path = '/Users/laurageneaulibourne/Downloads/S4/ia/exam/dataset/seg_train/seg_train'
dataset_test_path = '/Users/laurageneaulibourne/Downloads/S4/ia/exam/dataset/seg_test/seg_test'

classe = {'buildings' : 0, 'forest' : 1, 'glacier' : 2, 'mountain' : 3, 'sea' : 4, 'street' : 5}

for c in classe.keys():
    label_train = []
    label_test = []
    
    class_train_path = os.path.join(dataset_train_path, c)
    class_test_path = os.path.join(dataset_test_path, c)
    
    img_train_path = class_train_path +'/img'
    img_test_path = class_test_path+'/img'

    for imag in os.listdir(img_train_path):
        label_train.append({'image_name' : imag, 'label' : classe[c]})
        

    for imag in os.listdir(img_test_path):
        label_test.append({'image_name' : imag, 'label' : classe[c]})
        
        
    df_train = pd.DataFrame(label_train)
    df_train.to_csv(os.path.join(class_train_path, f"{classe[c]}_label_train.csv"), index=False)
    df_test = pd.DataFrame(label_test)
    df_test.to_csv(os.path.join(class_test_path, f"{classe[c]}_label_test.csv"), index=False)



Repartition of the dataset through the 6 classes 

create our dataset 

In [11]:
#the classes we are interested by
A = 'buildings'
B = 'forest'

path_img_A_train = os.path.join(dataset_train_path, A, 'img')
path_img_B_train = os.path.join(dataset_train_path, B, 'img')
path_img_AB_train = os.path.join(dataset_train_path, 'df/img')
path_img_A_test = os.path.join(dataset_test_path, A, 'img')
path_img_B_test = os.path.join(dataset_test_path, B, 'img')
path_img_AB_test = os.path.join(dataset_test_path, 'df/img')

#create the directory if it does not exist
os.makedirs(path_img_AB_train, exist_ok=True)
os.makedirs(path_img_AB_test, exist_ok=True)

def move(init, final):
    for file in os.listdir(init):
        source_file = os.path.join(init, file)
        dest_file = os.path.join(final, file)
        os.rename(source_file, dest_file)



move(path_img_A_train, path_img_AB_train)
move(path_img_B_train, path_img_AB_train)

move(path_img_A_test, path_img_AB_test)
move(path_img_B_test, path_img_AB_test)

label_A_train = pd.read_csv(os.path.join(dataset_train_path, A, f"{classe[A]}_label_train.csv"))
label_B_train = pd.read_csv(os.path.join(dataset_train_path, B, f"{classe[B]}_label_train.csv"))
label_AB_train = pd.concat([label_A_train, label_B_train], ignore_index=True)
label_AB_train.to_csv(os.path.join(dataset_train_path,'df', 'label_train.csv'), index=False)



label_A_test = pd.read_csv(os.path.join(dataset_test_path, A, f"{classe[A]}_label_test.csv"))
label_B_test = pd.read_csv(os.path.join(dataset_test_path, B, f"{classe[B]}_label_test.csv"))
label_AB_test = pd.concat([label_A_test, label_B_test], ignore_index=True)
label_AB_test.to_csv(os.path.join(dataset_test_path, 'df', 'label_test.csv'), index=False)

In [12]:
label_AB_train = pd.read_csv(os.path.join(dataset_train_path, 'df', 'label_train.csv'))
label_AB_test = pd.read_csv(os.path.join(dataset_test_path, 'df', 'label_test.csv'))

In [ ]:
class CustomImageDataset(Dataset):

    def __init__(self, annotations_file, img_dir, img_transform=None):

        self.img_labels = annotations_file 
        self.img_dir = img_dir
        self.img_transform = img_transform
        


    def __len__(self): #Return the number of samples in the dataset.
        return len(self.img_labels)
   
    def __getitem__(self, idx): #For a given index, it returns the sample (image, label) associated to it.
        
        # Get the file path of the image by combining the directory and the image filename from the labels DataFrame
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        
        # Read the image from the specified file path and convert it to a tensor
        image = read_image(img_path)
        
        # Retrieve the corresponding label for the image from the labels DataFrame
        label = self.img_labels.iloc[idx, 1]
        
        # If a transform function is provided, apply it to the image
        if self.transform:
            image = self.transform(image)
            # If a target_transform function is provided, apply it to the label
        
        # Return the transformed image and label as a tuple
        return image, label
            

In [16]:
# List of transformations that will be applied to images
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor() 
])

path_dataset ="/Users/laurageneaulibourne/Downloads/S4/computer_vision/exam/datasets/dataset_retina/Fundus_Image_Vessel_dataset"

dataset_train = CustomImageDataset(annotations_file= label_AB_train, img_dir = path_img_AB_train, img_transform=transform)
dataset_test = CustomImageDataset(annotations_file= label_AB_test, img_dir = path_img_AB_test, img_transform=transform)

train_dataloader = DataLoader(dataset_train, batch_size=64, shuffle=True)
test_dataloader= DataLoader(dataset_test, batch_size=64, shuffle=True)